# Ejercicio AirBnB: Extracción de entidades

En este cuaderno vamos a trabajar con un dataset de AirBnB de la ciudad de Oporto. Se puede encontrar más información sobre el dataset y otras implementaciones en [Porto](https://github.com/Vasallo94/Porto). 

El dataset contiene información sobre las características de las viviendas, su localización, el precio, el número de comentarios, etc.

El objetivo de este ejercicio es poder analizar los comentarios de los usuarios mediante LLMs y poder extraer información relevante de los mismos para su análisis posterior.

## Primera parte

Vamos a utilizar el archivo listings1_cleaned.csv que contiene información sobre las viviendas de AirBnB en Oporto.

In [ ]:
import pandas as pd
from google import genai
from IPython.display import display, Markdown
import json

# Configurar la API Key de Gemini o configura el cliente de groq! Usa el proveedor que quieras
GOOGLE_API_KEY = '----'  # Reemplaza con tu API Key de Google GenAI
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

data = pd.read_csv("docs/listings1_cleaned.csv")
data

,listing_id,listing_url,picture_url,name,description,host_id,host_name,host_since,host_response_rate,host_acceptance_rate,...,review_scores_rating,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,reviews_per_month,availability_365,has_availability,last_review,geographical_group
0,2.905900e+04,https://www.airbnb.com/rooms/29059,https://a0.muscache.com/pictures/736399/fa6c31...,Lovely studio Quartier Latin,CITQ 267153<br />Lovely studio with 1 closed r...,125031.0,Maryline,2010-05-14,100.000000,98.000000,...,4.670000,4.620000,4.810000,4.770000,4.820000,2.69000,302.0,t,2024-03-16,Central
1,2.906100e+04,https://www.airbnb.com/rooms/29061,https://a0.muscache.com/pictures/9e59d417-4b6a...,Maison historique - Quartier Latin,Lovely historic house with plenty of period ch...,125031.0,Maryline,2010-05-14,100.000000,98.000000,...,4.730000,4.660000,4.880000,4.810000,4.870000,0.88000,348.0,t,2024-02-19,Central
2,3.630100e+04,https://www.airbnb.com/rooms/36301,https://a0.muscache.com/pictures/26c20544-475f...,Romantic & peaceful Plateau loft,"Enjoy the best of Montreal in this romantic, ...",381468.0,Sylvie,2011-02-07,94.000000,80.000000,...,4.860000,4.860000,4.920000,4.900000,4.880000,0.47000,81.0,t,2024-01-07,Central
3,3.811800e+04,https://www.airbnb.com/rooms/38118,https://a0.muscache.com/pictures/213997/763ec1...,Beautiful room with a balcony in front of a parc,Nearest metro Papineau.,163569.0,M.,2010-07-11,78.000000,0.000000,...,4.500000,4.250000,4.810000,4.810000,4.630000,0.10000,299.0,t,2022-08-29,Central
4,5.047900e+04,https://www.airbnb.com/rooms/50479,https://a0.muscache.com/pictures/miso/Hosting-...,L'Arcade Douce,The appartement is sunny and ideally situated ...,231694.0,Noemie,2010-09-11,100.000000,100.000000,...,4.950000,4.940000,4.970000,4.980000,4.840000,1.60000,0.0,t,2024-03-18,South
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8135,1.116753e+18,https://www.airbnb.com/rooms/1116753448972266015,https://a0.muscache.com/pictures/miso/Hosting-...,"Logement ,1 chambre 04 personnes",If you are looking for a beautiful cozy and we...,429774011.0,Ammar,2021-10-31,96.922347,89.203748,...,4.715698,4.692737,4.814562,4.812302,4.768682,1.70759,257.0,t,NaN,North
8136,1.116820e+18,https://www.airbnb.com/rooms/1116820020797670135,https://a0.muscache.com/pictures/miso/Hosting-...,Superbe maison à Montréal,Beautiful entire place in the heart of Montreal,126764590.0,Samuel,2017-04-20,96.922347,89.203748,...,4.715698,4.692737,4.814562,4.812302,4.768682,1.70759,269.0,t,NaN,East
8137,1.117028e+18,https://www.airbnb.com/rooms/1117028063118176710,https://a0.muscache.com/pictures/miso/Hosting-...,Luxurious 2BR - Downtown Montreal,Feel at home at these brand new construction C...,159008278.0,Melissa,2017-11-16,100.000000,100.000000,...,4.715698,4.692737,4.814562,4.812302,4.768682,1.70759,156.0,t,NaN,Central
8138,1.117273e+18,https://www.airbnb.com/rooms/1117273423317641658,https://a0.muscache.com/pictures/miso/Hosting-...,Magnifique appart Plateau,"Fully renovated condo style apartment, beautif...",214303569.0,Jean-Georges,2018-09-08,100.000000,93.000000,...,4.715698,4.692737,4.814562,4.812302,4.768682,1.70759,192.0,t,NaN,Central


1. Filtra el dataset con una función que seleccione las 10/20 reseñas con mayor longitud (de string) en la columna descripción (las que consideraríamos las más 'relevantes')
2. Elabora un prompt de tal forma que un LLM sea capaz de extraer un json con las siguientes entidades de las entradas de la tabla filtrada:
```json
{
    "name": str,
    "location": str,
    "main_characteristics": str,
    "type": str,
    "size": str,
    "capacity": str,
    "key_amenities": list,
    "proximity_highlights": list,
}
```
3. Puedes elegir el proveedor que quieras, Gemini de Google o cualquiera de los modelos de Groq

## Parte 2

Realiza un análisis de los comentarios de los apartamentos con un LLM y extrae información relevante de los mismos.

Por ejemplo, puedes analizar los comentarios y extraer información sobre la limpieza, la ubicación, la relación calidad-precio, el sentimiento del comentario, etc. 

Igual que la parte 1 pero con un segundo dataset y con formato de salida a tu gusto

In [3]:
comentarios = pd.read_csv("Airbnb_reviews_5000.csv")
comentarios

,name,host_id,host_name,date,reviewer_id,reviewer_name,comments,language
0,LUXURY apartment t3 oporto antas,37249350.0,Joaquim,2018-01-30,156292241.0,Silvia,Acomodação espaçosa e bonita. O Joaquim e sua ...,pt
1,Aida's Haven | Room&PrivateBath | St. Catarina,38365612.0,Alexandra,2019-12-15,283158910.0,Tierry Dayan,"A anfitriã Alexandra foi muito simpática, pre...",pt
2,Maritime Inspiration - One Bedroom Beach Apart...,147469727.0,Susana,2018-08-12,173756309.0,Julio,"Apartamento muy bien comunicado, aunque nos es...",es
3,Central charming Top floor - nice views,26222276.0,A.Maria,2016-05-28,11705876.0,Rafael,"Adosinda was kind, showed the apartment, set u...",en
4,Ribeira Oporto Apartment II (Renewed 2021),35057317.0,João,2016-06-02,38269559.0,Anais,"Un appartement au coeur de porto, très grand a...",fr
...,...,...,...,...,...,...,...,...
4995,Marquês's House - Estúdio ao Marquês,93991335.0,Conceição,2017-09-10,13710198.0,Sandra,Merci pour le chaleureux accueil et tous les b...,fr
4996,M1.4 | Surf Beach Matosinhos | Porto,39551356.0,Porto Je T'Aime,2018-08-04,204048434.0,Louis,Toutes les qualités recherchées dans ce type d...,fr
4997,Kitchnette Studio in Porto's Downtown II,1651078.0,Lurdes,2022-06-19,68564036.0,Angie,"Its a lovely little studio, a green oasis, rig...",en
4998,MyLoft-Charming Apartment!!! - Metro Bolhão,34348897.0,Carla,2022-06-23,283375850.0,Angelika,Carla ist wirklich eine sehr tolle Gastgeberin...,de
